# Estudio de ablación — `50mil_tarjetas_3_al_31_marzo`

## ¿Qué es un estudio de ablación?

Un estudio de ablación cuantifica la contribución real de cada módulo o decisión de diseño del modelo.
Se parte de la **configuración óptima** (trial #131 de `mdlf_marzo2025_v3_line`, Top-1 = 0.7134 con 30 épocas)
y se ejecutan variantes donde se **elimina o cambia un elemento a la vez**, manteniendo todo lo demás fijo.
La diferencia en Top-1 respecto al baseline mide el impacto real de ese componente.

Responde a: **¿cuánto aporta realmente LINE, LDA, `include_prox`, `neighborhood_crowd` y AdamW?**

## Experimentos

| # | Nombre | Cambio respecto al baseline |
|---|---|---|
| 0 | `baseline` | Ninguno — resultado de referencia |
| 1 | `no_line` | `ablation={'line'}` — embeddings LINE sustituidos por ceros |
| 2 | `no_lda` | `ablation={'lda'}` — vectores de tópicos LDA sustituidos por ceros |
| 3 | `no_prox` | `include_prox=False` — sin features de proximidad espacial |
| 4 | `no_neighborhood_crowd` | `ablation={'neighborhood_crowd'}` — sin demanda colectiva de estaciones vecinas |
| 5 | `sgd` | `optimizer='sgd'` — mismo `lr` que el best trial (tuneado para AdamW) |

## Notas de implementación

- **`ablation`**: `PipelineMDLF` acepta un `set` de strings. Los módulos ablados reciben arrays de **ceros**,
  no se eliminan del grafo CNN. El modelo sigue entrenándose con la misma arquitectura.
- **`include_prox=False`**: no existe valor `'prox'` en `ablation`; se pasa directamente como parámetro.
- **`optimizer='sgd'`**: se mantiene el mismo `lr` del best trial. La comparación no es completamente justa
  (el `lr` fue tuneado para AdamW), pero mide el impacto del optimizador sin re-optimizar el `lr`.
  Si `delta_top1` es grande, confirma que AdamW es relevante incluso con el lr del contrincante.


In [ ]:
import os, sys
from pathlib import Path

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF').resolve()
_estudios = _cubo_dir.parent

os.chdir(_cubo_dir)
for _p in [str(_cubo_dir), str(_mdlf_dir), str(_estudios)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from pipeline_mdlf import PipelineMDLF

print(f'CWD      → {os.getcwd()}')
print(f'MDLF dir → {_mdlf_dir}')

In [ ]:
# ── Constantes ────────────────────────────────────────────────────────────────

ALMACEN    = '50mil_tarjetas_3_al_31_marzo'
SPLIT_DATE = '2025-03-24'
N_EPOCHS   = 30

LINE_PATH = _mdlf_dir / 'runs' / 'mdlf_marzo2025_v3_line_line_best.parquet'
assert LINE_PATH.exists(), f'LINE parquet no encontrado: {LINE_PATH}'
print(f'LINE parquet : {LINE_PATH.name}  ✓')

_descomp = _estudios / 'descomprimido'
TTP_PATHS = [
    str(_descomp / 'TTP_PERIODO_01_2025.csv'),
    str(_descomp / 'TTP_PERIODO_02_2025.csv'),
    str(_descomp / 'TTP_PERIODO_03_2025.csv'),
]

OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

# Hiperparámetros óptimos — trial #131, mdlf_marzo2025_v3_line
# top1_proxy (7 épocas) = 0.7221 | top1_final (30 épocas) = 0.7134
BEST_PARAMS = {
    # LINE (no se pasan a PipelineMDLF, el parquet ya está entrenado)
    'line_dim'          : 16,
    'line_asymmetric'   : True,
    'line_num_negative' : 5,
    # LDA
    'n_topics_cercanias'     : 8,
    'lda_alpha'              : 0.00353564483968244,
    'lda_beta_none'          : True,     # beta=None
    'lda_log_counts'         : True,
    'lda_min_df'             : 1,
    'lda_min_station_trips'  : 100,
    # LDA fijos (15/15 top-v2)
    'lda_use_periods'        : False,
    'lda_token_granularity'  : 'temporal',
    # CNN
    'n_heads'      : 2,
    'd_t'          : 8,
    'cnn_channels' : 32,
    'kernel_size'  : 3,          # fijo
    # Entrenamiento
    'optimizer'    : 'adamw',
    'lr'           : 0.004790657275275601,
    'weight_decay' : 0.06524408971165131,
    'lr_scheduler' : True,       # fijo
    'n_weeks'      : 4,
    # Extensiones
    'include_prox'               : True,  # fijo
    'include_neighborhood_crowd' : True,
    'norm'                       : 'none',
}

print(f'Almacén    : {ALMACEN}')
print(f'Split date : {SPLIT_DATE}')
print(f'N_epochs   : {N_EPOCHS}')

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────

def run_ablation(name: str, **overrides) -> dict:
    """
    Ejecuta PipelineMDLF con BEST_PARAMS + overrides.
    Devuelve dict con name, top1, top5.
    """
    params = {**BEST_PARAMS, **overrides}

    lda_beta = None if params.pop('lda_beta_none', True) else params.pop('lda_beta', None)
    norm_val = params.pop('norm', 'none')
    norm = None if norm_val == 'none' else norm_val

    # Separar parámetros que no van directo a PipelineMDLF
    _skip = {'lda_beta', 'lda_beta_none', 'norm',
              'line_dim', 'line_asymmetric', 'line_num_negative'}
    pipe_kwargs = {k: v for k, v in params.items() if k not in _skip}

    pipe = PipelineMDLF(
        nombre_almacen=ALMACEN,
        ttp_paths=TTP_PATHS,
        od_matrices_subdirs=OD_MATRICES_SUBDIRS,
        line_ap_path=LINE_PATH,
        split_date=SPLIT_DATE,
        lda_beta=lda_beta,
        norm=norm,
        n_epochs=N_EPOCHS,
        random_state=42,
        **pipe_kwargs,
    )

    pipe.run(run_name=f'ablacion_{name}')
    result = {'name': name, 'top1': pipe.test_accuracy, 'top5': pipe.test_accuracy5}
    print(f'  [{name}]  Top-1={result["top1"]:.4f}  Top-5={result["top5"]:.4f}')
    return result

In [ ]:
# ── Exp 0: Baseline ───────────────────────────────────────────────────────────
# Todos los módulos activos. Debe reproducir top1 ≈ 0.7134 (30 épocas).
# Puede variar levemente por la estocasticidad del CNN (random_state=42).

results = []
results.append(run_ablation('baseline'))

In [ ]:
# ── Exp 1: Sin LINE ───────────────────────────────────────────────────────────
# Los embeddings LINE de cada estación se sustituyen por un vector de ceros.
# Si delta_top1 < 0 y significativo: LINE aporta información real sobre la
# similitud funcional entre estaciones del grafo de cercanías.

results.append(run_ablation('no_line', ablation={'line'}))

In [ ]:
# ── Exp 2: Sin LDA ────────────────────────────────────────────────────────────
# Los vectores de distribución de tópicos LDA (theta) se sustituyen por ceros.
# Si delta_top1 < 0: el modelado de función de región (qué tipo de zona es
# cada estación) ayuda a predecir el destino.

results.append(run_ablation('no_lda', ablation={'lda'}))

In [ ]:
# ── Exp 3: Sin include_prox ───────────────────────────────────────────────────
# Se eliminan las features de proximidad espacial (distancia entre estación
# de subida y candidatos de destino).
# Nota: 'prox' no es un valor válido en ablation={...}; se pasa directamente
# como include_prox=False porque es un parámetro de construcción de features.

results.append(run_ablation('no_prox', include_prox=False))

In [ ]:
# ── Exp 4: Sin neighborhood_crowd ────────────────────────────────────────────
# Las features de demanda colectiva de estaciones vecinas se sustituyen por ceros.
# Mide el aporte de saber cuánta gente viaja hacia cada destino desde zonas
# cercanas a la estación de subida del usuario.

results.append(run_ablation('no_neighborhood_crowd', ablation={'neighborhood_crowd'}))

In [ ]:
# ── Exp 5: SGD en lugar de AdamW ─────────────────────────────────────────────
# Se sustituye AdamW por SGD manteniendo el mismo lr (0.00479), que fue
# optimizado para AdamW. La comparación no es del todo justa (SGD suele
# necesitar un lr diferente), pero mide el impacto del optimizador sin
# reoptimizar. Un delta grande confirma que AdamW es relevante.

results.append(run_ablation('sgd', optimizer='sgd'))

In [ ]:
# ── Exp 6: Sin include_prox + sin neighborhood_crowd ─────────────────────────
# Desactiva simultáneamente ambos módulos de señal contextual externa.
# Permite ver el impacto combinado y si sus contribuciones son independientes.

results.append(run_ablation(
    'sin_prox_sin_neighborhood',
    include_prox=False,
    include_neighborhood_crowd=False,
))

In [ ]:
# ── Tabla de resultados ───────────────────────────────────────────────────────

import pandas as pd

df_results = pd.DataFrame(results)

baseline_top1 = df_results.loc[df_results['name'] == 'baseline', 'top1'].values[0]
baseline_top5 = df_results.loc[df_results['name'] == 'baseline', 'top5'].values[0]

df_results['delta_top1'] = df_results['top1'] - baseline_top1
df_results['delta_top5'] = df_results['top5'] - baseline_top5

df_results = df_results.sort_values('top1', ascending=False).reset_index(drop=True)

print(f'Baseline → Top-1={baseline_top1:.4f}  Top-5={baseline_top5:.4f}')
print()

display(
    df_results.style.format({
        'top1'      : '{:.4f}',
        'top5'      : '{:.4f}',
        'delta_top1': '{:+.4f}',
        'delta_top5': '{:+.4f}',
    }).background_gradient(
        subset=['delta_top1'], cmap='RdYlGn', vmin=-0.05, vmax=0.01
    )
)